<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/vision-language-composition/blob/main/scripts/MACCO_Table1_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACCO Table 1
This notebook evaluates the released **MACCO ViT-B/32**  andour fine-tuned **MACCO-W1-CML-ViT-B-32** model. No training, Drive, GitHub repository downloads or project setup.

The final table compares the nine MACCO entries with the paper. Other methods are shown as paper references only. VL-Checklist requires large image archives; they are streamed and only referenced images are retained on `/content`. Network transfer can still be large. Full GPU reproduction has not been run by the notebook author.


## 1 Install the necessary packages


In [1]:
%pip -q install open_clip_torch==2.32.0 huggingface_hub pandas requests spacy==3.8.16 nltk==3.9.1



In [2]:
import os, io, json, tarfile, random, re, shutil
from pathlib import Path
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor
import requests, numpy as np, pandas as pd, torch, open_clip
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import HfApi, hf_hub_download, hf_hub_url
from tqdm.auto import tqdm

DATA = Path('/content/macco_data')
DATA.mkdir(exist_ok=True)
DEVICE = 'cuda'


## 2 Load the trained model (OG MACCO)


In [ ]:
checkpoint = hf_hub_download('hiker-lw/MACCO', 'MACCO-CLIP-ViT-B-32.pt')
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=checkpoint, device=DEVICE, precision='fp32')
model.eval()
tokenizer = open_clip.tokenize
print('MACCO loaded.')


MACCO-CLIP-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

MACCO-CLIP-ViT-B-32.pt: downloading bytes:           |  0.00B            

MACCO loaded.


## 2.1 Our fine-tuned model (Run this to evaluate our fine tuned model)

In [3]:
checkpoint = hf_hub_download('rifahnanjiba2/MACCO-Finetuned', 'MACCO-W1-CML-ViT-B-32.pt')
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=checkpoint, device=DEVICE, precision='fp32')
model.eval()
tokenizer = open_clip.tokenize

MACCO-W1-CML-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

MACCO-W1-CML-ViT-B-32.pt: downloading bytes:           |  0.00B            

## 3 Table 1 reference values


In [4]:
columns = ['ARO Relation', 'ARO Attribute', 'ARO Order',
           'Sugar-Crepe Relation', 'Sugar-Crepe Attribute',
           'VL-Checklist Relation', 'VL-Checklist Attribute',
           'VALSE Relation', "What's-up"]
paper = pd.DataFrame({
    'CLIP': [58.7,62.7,54.1,68.8,70.8,63.6,67.7,70.1,41.8],
    'CLIP-FT': [64.3,66.2,49.1,71.1,77.7,60.9,67.4,69.3,41.4],
    'IL-CLIP': [50.0,55.3,16.7,56.3,63.9,55.7,59.5,55.7,42.3],
    'SDS-CLIP': [53.0,62.0,29.0,*([np.nan]*6)],
    'CLIP-CAE': [69.5,65.4,np.nan,73.0,78.9,65.4,68.6,68.8,np.nan],
    'MACCO': [73.1,68.5,76.0,77.1,79.1,70.2,68.7,75.3,43.2]
}, index=columns)
results = {}
display(paper)  # Published values only; our measurements come below.


,CLIP,CLIP-FT,IL-CLIP,SDS-CLIP,CLIP-CAE,MACCO
ARO Relation,58.7,64.3,50.0,53.0,69.5,73.1
ARO Attribute,62.7,66.2,55.3,62.0,65.4,68.5
ARO Order,54.1,49.1,16.7,29.0,NaN,76.0
Sugar-Crepe Relation,68.8,71.1,56.3,NaN,73.0,77.1
Sugar-Crepe Attribute,70.8,77.7,63.9,NaN,78.9,79.1
VL-Checklist Relation,63.6,60.9,55.7,NaN,65.4,70.2
VL-Checklist Attribute,67.7,67.4,59.5,NaN,68.6,68.7
VALSE Relation,70.1,69.3,55.7,NaN,68.8,75.3
What's-up,41.8,41.4,42.3,NaN,NaN,43.2


In [5]:
def show_table():
    table = pd.DataFrame({'Paper MACCO': paper['MACCO']})
    table['Our MACCO'] = pd.Series(results, dtype=float)
    table['Difference'] = table['Our MACCO'] - table['Paper MACCO']
    return table

def record(name, fraction):
    results[name] = float(fraction) * 100
    print(f'{name}: {results[name]:.2f}')
    show_table().to_csv('/content/MACCO_Table1.csv')


## 4 Small shared functions
These functions read data and score image–caption matches. They do not train anything.


In [6]:
DATASET = 'hiker-lw/VL-Compositionality-Benchmarks'
api = HfApi()

def dataset_url(name):
    return hf_hub_url(DATASET, name, repo_type='dataset')

def extract(name, keep=None):
    with requests.get(dataset_url(name), stream=True, timeout=120) as response:
        response.raise_for_status()
        with tarfile.open(fileobj=response.raw, mode='r|*') as archive:
            for member in archive:
                if keep is None or keep(member.name):
                    archive.extract(member, DATA, filter='data')
    print('Extracted:', name)

def load_json(path):
    return json.loads(Path(path).read_text())

def folder(marker):
    return next(DATA.rglob(marker)).parent


In [7]:
@lru_cache(None)
def image_index(root):
    index = {}
    for path in Path(root).rglob('*'):
        if path.suffix.lower() in ['.jpg','.jpeg','.png']:
            index.setdefault(path.name, []).append(path)
    return index

def image_path(root, relative):
    path = Path(root) / relative
    if path.is_file(): return path
    matches = image_index(str(root)).get(Path(relative).name, [])
    if len(matches) == 1: return matches[0]
    raise FileNotFoundError(f'Missing or ambiguous image: {path}')

def row(root, relative, captions, target=1, crop=None):
    return (str(image_path(root, relative)), tuple(captions), target, crop)


In [8]:
class Examples(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        path, captions, target, crop = self.rows[index]
        with Image.open(path) as original:
            image = original.convert('RGB')
            if crop is not None: image = image.crop(crop)
            image = preprocess(image)
        return image, captions, target


In [9]:
@torch.no_grad()
def evaluate(rows):
    loader = DataLoader(Examples(rows), batch_size=32,
                        num_workers=2, shuffle=False)
    correct = []
    for images, caption_options, targets in tqdm(loader):
        images = model.encode_image(images.to(DEVICE)).cpu().numpy()
        images /= np.linalg.norm(images, axis=1, keepdims=True)
        texts = []
        for option in caption_options:
            tokens = tokenizer(list(option)).to(DEVICE)
            features = model.encode_text(tokens).cpu().numpy()
            features /= np.linalg.norm(features, axis=1, keepdims=True)
            texts.append(features)
        scores = np.einsum('bd,bkd->bk', images, np.stack(texts, axis=1))
        correct.extend(scores.argmax(axis=1) == targets.numpy())
    return np.asarray(correct, dtype=bool)


## 5 Sugar-Crepe


In [ ]:
extract('sugar-crepe.tar.gz')
sugar = folder('replace_rel.json')
sugar_scores = {}


Extracted: sugar-crepe.tar.gz


In [ ]:
for name in ['replace_rel','add_att','replace_att','swap_att']:
    annotations = list(load_json(sugar / f'{name}.json').values())
    rows = [row(sugar/'val2017', x['filename'],
                [x['negative_caption'], x['caption']]) for x in annotations]
    sugar_scores[name] = evaluate(rows).mean()
    print(name, round(sugar_scores[name]*100, 2))

record('Sugar-Crepe Relation', sugar_scores['replace_rel'])
record('Sugar-Crepe Attribute', np.mean(
    [sugar_scores[k] for k in ['add_att','replace_att','swap_att']]))


  0%|          | 0/44 [00:00<?, ?it/s]

replace_rel 76.96


  0%|          | 0/22 [00:00<?, ?it/s]

add_att 82.51


  0%|          | 0/25 [00:00<?, ?it/s]

replace_att 84.52


  0%|          | 0/21 [00:00<?, ?it/s]

swap_att 70.72
Sugar-Crepe Relation: 76.96
Sugar-Crepe Attribute: 79.25


## 6 ARO Relation and Attribute


In [ ]:
extract('ARO.tar.gz')
aro = folder('visual_genome_relation.json')

def aro_rows(annotations):
    rows = []
    for x in annotations:
        box = (x['bbox_x'], x['bbox_y'],
               x['bbox_x']+x['bbox_w'], x['bbox_y']+x['bbox_h'])
        rows.append(row(aro/'images', x['image_path'],
                        [x['false_caption'], x['true_caption']], crop=box))
    return rows


Extracted: ARO.tar.gz


The next short data cell preserves the authors’ ARO Relation category exclusions.


In [ ]:
excluded_relations = {
    'leaning against', 'pulled by', 'pulling', 'adjusting', 'attached to', 'between', 'bigger than', 'biting', 'boarding', 'brushing',
    'chewing', 'cleaning', 'climbing', 'close to', 'coming from', 'coming out of', 'contain', 'crossing', 'dragging', 'draped over',
    'drinking', 'drinking from', 'driving', 'driving down', 'driving on', 'eating from', 'eating in', 'enclosing', 'exiting', 'facing',
    'filled with', 'floating in', 'floating on', 'flying', 'flying above', 'flying in', 'flying over', 'flying through', 'full of', 'going down',
    'going into', 'going through', 'grazing in', 'growing in', 'growing on', 'guiding', 'hanging from', 'hanging in', 'hanging off', 'hanging over',
    'higher than', 'holding onto', 'hugging', 'in between', 'jumping off', 'jumping on', 'jumping over', 'kept in', 'larger than', 'leading',
    'leaning over', 'leaving', 'licking', 'longer than', 'looking in', 'looking into', 'looking out', 'looking over', 'looking through', 'lying next to',
    'lying on top of', 'making', 'mixed with', 'mounted on', 'moving', 'on the back of', 'on the edge of', 'on the front of', 'on the other side of', 'opening',
    'painted on', 'parked at', 'parked beside', 'parked by', 'parked in', 'parked in front of', 'parked near', 'parked next to', 'perched on', 'petting',
    'piled on', 'playing', 'playing in', 'playing on', 'playing with', 'pouring', 'reaching for', 'reading', 'reflected on', 'riding on',
    'running in', 'running on', 'running through', 'seen through', 'sitting behind', 'sitting beside', 'sitting by', 'sitting in front of', 'sitting near', 'sitting next to',
    'sitting under', 'skiing down', 'skiing on', 'sleeping in', 'sleeping on', 'smiling at', 'sniffing', 'splashing', 'sprinkled on', 'stacked on',
    'standing against', 'standing around', 'standing behind', 'standing beside', 'standing in front of', 'standing near', 'standing next to', 'staring at', 'stuck in', 'surrounding',
    'swimming in', 'swinging', 'talking to', 'topped with', 'touching', 'traveling down', 'traveling on', 'tying', 'typing on', 'underneath',
    'wading in', 'waiting for', 'walking across', 'walking by', 'walking down', 'walking next to', 'walking through', 'working in', 'working on', 'worn on',
    'wrapped around', 'wrapped in', 'by', 'of', 'near', 'next to', 'with', 'beside', 'on the side of', 'around',
}


In [ ]:
annotations = load_json(aro/'visual_genome_relation.json')
correct = evaluate(aro_rows(annotations))
groups = pd.DataFrame({'category': [x['relation_name'] for x in annotations],
                       'correct': correct}).groupby('category').correct.mean()
groups = groups[~groups.index.isin(excluded_relations)].round(4)
record('ARO Relation', groups.mean())


  0%|          | 0/749 [00:00<?, ?it/s]

ARO Relation: 73.07


In [ ]:
annotations = load_json(aro/'visual_genome_attribution.json')
correct = evaluate(aro_rows(annotations))
groups = pd.DataFrame({
    'category': [f"{x['attributes'][0]}_{x['attributes'][1]}" for x in annotations],
    'correct': correct}).groupby('category').correct.agg(['mean','count'])
record('ARO Attribute', groups.loc[groups['count'] >= 25, 'mean'].round(4).mean())


  0%|          | 0/899 [00:00<?, ?it/s]

ARO Attribute: 68.99


## 7 ARO Order
The ARO archive includes Flickr images. Only the COCO test images referenced by the annotations are downloaded below, if missing.


In [ ]:
!python -m spacy download en_core_web_sm --quiet
import spacy, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nlp = spacy.load('en_core_web_sm')
coco_annotations = load_json(folder('coco_karpathy_test.json')/'coco_karpathy_test.json')
flickr_annotations = load_json(folder('flickr30k_test.json')/'flickr30k_test.json')
coco = DATA/'coco'
coco.mkdir(exist_ok=True)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
wanted = {int(Path(x['image']).stem.split('_')[-1]): x['image']
          for x in coco_annotations}

for n in tqdm(range(5), desc='COCO test shards'):
    url = hf_hub_url('undefined443/coco-karpathy-wds',
                     f'test/test-{n:05d}.tar', repo_type='dataset')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tarfile.open(fileobj=response.raw, mode='r|*') as archive:
            pairs = {}
            for member in archive:
                suffix = Path(member.name).suffix
                if not member.isfile() or suffix not in ('.jpg', '.json'):
                    continue
                key = Path(member.name).stem
                pair = pairs.setdefault(key, {})
                pair[suffix] = archive.extractfile(member).read()
                if len(pair) < 2:
                    continue
                image_id = int(json.loads(pair['.json'])['cocoid'])
                if image_id in wanted:
                    output = coco / wanted[image_id]
                    output.parent.mkdir(parents=True, exist_ok=True)
                    if not output.is_file():
                        output.write_bytes(pair['.jpg'])
                del pairs[key]

missing = [x['image'] for x in coco_annotations
           if not (coco / x['image']).is_file()]
if missing:
    raise RuntimeError(f'{len(missing)} COCO images missing: {missing[:3]}')
print('COCO images ready:', len(coco_annotations))

COCO test shards:   0%|          | 0/5 [00:00<?, ?it/s]

COCO images ready: 5000


In [ ]:
def clean_caption(text):
    text = re.sub(r'([.!"()*#:;~])', ' ', text.lower())
    text = re.sub(r'\s{2,}', ' ', text).rstrip('\n').strip(' ')
    return ' '.join(text.split(' ')[:30])

def shuffle_pos(text, selected):
    tokens = list(nlp(text))
    words = np.array([t.text for t in tokens])
    nouns = [i for i,t in enumerate(tokens) if t.tag_ in ['NN','NNS','NNP','NNPS']]
    adjectives = [i for i,t in enumerate(tokens) if t.tag_ in ['JJ','JJR','JJS']]
    groups = [nouns, adjectives] if selected else [
        [i for i in range(len(tokens)) if i not in nouns+adjectives]]
    for indices in groups: words[indices] = np.random.permutation(words[indices])
    return ' '.join(words)


In [ ]:
def shuffle_trigrams(text, inside):
    words = nltk.word_tokenize(text)
    groups = [words[i:i+3] for i in range(0,len(words),3)]
    if inside:
        for group in groups: random.shuffle(group)
    else: random.shuffle(groups)
    return ' '.join(' '.join(g) for g in groups)

def order_rows(annotations, root):
    rows = []
    for x in tqdm(annotations, desc='Preparing captions'):
        for text in x['caption']:
            options = [text, shuffle_pos(text, True), shuffle_pos(text, False),
                       shuffle_trigrams(text, True), shuffle_trigrams(text, False)]
            rows.append(row(root, x['image'], [clean_caption(t) for t in options], 0))
    return rows


In [ ]:
random.seed(0); np.random.seed(0)
coco_order = evaluate(order_rows(coco_annotations, coco)).mean()
random.seed(0); np.random.seed(0)
flickr_root = aro/'flickr30k-images'
flickr_order = evaluate(order_rows(flickr_annotations, flickr_root)).mean()
print('COCO Order:', round(coco_order*100, 2))
print('Flickr Order:', round(flickr_order*100, 2))
record('ARO Order', (coco_order + flickr_order)/2)


Preparing captions:   0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

Preparing captions:   0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/157 [00:00<?, ?it/s]

COCO Order: 75.52
Flickr Order: 81.92
ARO Order: 78.72


## 8 What’s-up
Use both VG/GQA subsets, then average the controlled, COCO and GQA group means.


In [ ]:
extract('whats_up_data.tar.gz')
spatial = folder('controlled_images_dataset.json')
controlled_scores = []
for name in ['controlled_images_dataset.json','controlled_clevr_dataset.json']:
    annotations = load_json(spatial/name)
    rows = [row(spatial, x['image_path'], x['caption_options'], 0) for x in annotations]
    controlled_scores.append(evaluate(rows).mean())


Extracted: whats_up_data.tar.gz


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f0a13f93100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f0a13f93100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

In [ ]:
spatial_groups = {'Controlled': np.mean(controlled_scores)}
for prefix, label, image_folder in [('coco','COCO','val2017'),('vg','GQA','vg_images')]:
    values = []
    for subset in ['one','two']:
        annotations = load_json(spatial/f'{prefix}_qa_{subset}_obj.json')
        rows = []
        for x in annotations:
            filename = f'{int(x[0]):012d}.jpg' if prefix=='coco' else f'{x[0]}.jpg'
            rows.append(row(spatial/image_folder, filename, [x[1],x[2]], 0))
        values.append(evaluate(rows).mean())
    spatial_groups[label] = np.mean(values)
print({k: round(v*100,2) for k,v in spatial_groups.items()})
record("What's-up", np.mean(list(spatial_groups.values())))


  0%|          | 0/71 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f0a13f93100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f0a13f93100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

  0%|          | 0/37 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

{'Controlled': np.float64(35.5), 'COCO': np.float64(47.21), 'GQA': np.float64(46.94)}
What's-up: 43.22


## 9 VL-Checklist and VALSE annotations
VALSE’s unused image tasks are not extracted. The required action images are shared with VL-Checklist.


In [10]:
valse_names = ['actant-swap.json','action-replacement.json','relations.json']
extract('VALSE.tar.gz', keep=lambda name: Path(name).name in valse_names)
valse_root = folder('actant-swap.json')
valse = {name: load_json(valse_root/name) for name in valse_names}
vl = DATA/'VL_checklist'
vl.mkdir(exist_ok=True)
vl_annotations = {}
relation_names = ['hake_action.json','swig_action.json','vg_action.json','vg_spatial.json']
attribute_names = [f'{g}/{a}.json' for g in ['vaw','vg']
                   for a in ['action','color','material','size','state']]



Extracted: VALSE.tar.gz


In [11]:
for name, data in valse.items():
    valse[name] = list(data.values()) if isinstance(data, dict) else data
    for item in valse[name]:
        if 'image_path' not in item:
            group = {'SWiG': 'swig', 'coco_2017': 'val2017'}[item['dataset']]
            item['image_path'] = f"{group}/{item['image_file']}"
        if 'pos_caption' not in item: item['pos_caption'] = item['caption']
        if 'neg_caption' not in item: item['neg_caption'] = item['foil']
print({name: len(data) for name, data in valse.items()})


{'actant-swap.json': 949, 'action-replacement.json': 648, 'relations.json': 535}


In [12]:
for kind, names in [('Relation',relation_names), ('Attribute',attribute_names)]:
    for name in names:
        remote = f'VL_checklist/VL_checklist_json_data/{kind}/{name}'
        response = requests.get(dataset_url(remote), timeout=120)
        response.raise_for_status()
        vl_annotations[f'{kind}/{name}'] = response.json()

needed = {group: set() for group in ['hake','swig','vg']}
for key, annotations in vl_annotations.items():
    group = key.split('/')[1].split('_')[0] if key.startswith('Relation/') else 'vg'
    needed[group].update(x['image_path'] for x in annotations)
for name in valse_names[:2]:
    for x in valse[name]:
        group, relative = x['image_path'].split('/',1)
        needed[group].add(relative)


## 10 Stream only the needed VL-Checklist images
These two short helper cells read split archives without storing archive copies.


In [13]:
class Parts:
    def __init__(self, filenames):
        self.names = iter(filenames); self.response = None
    def read(self, size):
        output = bytearray()
        while len(output) < size:
            if self.response is None:
                name = next(self.names, None)
                if name is None: break
                self.response = requests.get(dataset_url(name), stream=True, timeout=120)
                self.response.raise_for_status()
            chunk = self.response.raw.read(size-len(output))
            if chunk: output.extend(chunk)
            else: self.response.close(); self.response = None
        return bytes(output)
    def close(self):
        if self.response is not None: self.response.close()


In [14]:
def extract_selected(parts, destination, wanted):
    by_name = {}
    for relative in wanted: by_name.setdefault(Path(relative).name, []).append(relative)
    stream = Parts(parts)
    try:
        with tarfile.open(fileobj=stream, mode='r|*') as archive:
            for member in tqdm(archive, desc=destination.name):
                for relative in by_name.get(Path(member.name).name, []):
                    if member.isfile() and member.name.endswith('/'+relative):
                        output = destination/relative
                        output.parent.mkdir(parents=True, exist_ok=True)
                        with archive.extractfile(member) as source, output.open('wb') as target:
                            shutil.copyfileobj(source, target)
                        wanted.discard(relative)
                if not wanted: break
    finally: stream.close()
    if wanted: raise FileNotFoundError(f'{len(wanted)} required images absent: {list(wanted)[:3]}')


In [15]:
files = api.list_repo_files(DATASET, repo_type='dataset')
vl_images = vl/'VL_checklist_datasets'
for group, wanted in needed.items():
    destination = vl_images/group
    destination.mkdir(parents=True, exist_ok=True)
    wanted = {name for name in wanted if not (destination/name).is_file()}
    if group=='vg' and 'aro' in globals() and (aro/'images').is_dir():
        for name in list(wanted):
            matches = image_index(str(aro/'images')).get(Path(name).name, [])
            if len(matches)==1:
                target = destination/name
                target.parent.mkdir(parents=True, exist_ok=True)
                os.link(matches[0],target); wanted.remove(name)
    parts = sorted(f for f in files if f.startswith(
        f'VL_checklist/VL_checklist_datasets/{group}.tar.gz.part-'))
    if wanted: extract_selected(parts, destination, wanted)


hake: 0it [00:00, ?it/s]

swig: 0it [00:00, ?it/s]

vg: 0it [00:00, ?it/s]

## 11 VL-Checklist scores


In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True  # matches the authors' VL-Checklist evaluator
vl_relation = {}
for name in relation_names:
    annotations = vl_annotations['Relation/'+name]
    root = vl_images/name.split('_')[0]
    rows = [row(root, x['image_path'], [x['NEG'],x['POS']]) for x in annotations]
    vl_relation[name] = evaluate(rows).mean()
    print(name, round(vl_relation[name]*100,2))
action = np.mean([vl_relation[n] for n in relation_names[:3]])
record('VL-Checklist Relation', (action+vl_relation['vg_spatial.json'])/2)


  0%|          | 0/6313 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


hake_action.json 85.41


  0%|          | 0/786 [00:00<?, ?it/s]

swig_action.json 76.65


  0%|          | 0/398 [00:00<?, ?it/s]

vg_action.json 64.93


  0%|          | 0/958 [00:00<?, ?it/s]

vg_spatial.json 65.27
VL-Checklist Relation: 70.47


# **28/10/2026**
# *comment: ran upto this so far*

In [16]:
vl_attribute = {}
for name in attribute_names:
    annotations = vl_annotations['Attribute/'+name]
    rows = [row(vl_images/'vg', x['image_path'], [x['NEG'],x['POS']]) for x in annotations]
    vl_attribute[name] = evaluate(rows).mean()
    print(name, round(vl_attribute[name]*100,2))
types = ['action','color','material','size','state']
means = [np.mean([vl_attribute[f'vaw/{t}.json'],vl_attribute[f'vg/{t}.json']]) for t in types]
record('VL-Checklist Attribute', np.mean(means))


  0%|          | 0/95 [00:00<?, ?it/s]

vaw/action.json 75.35


  0%|          | 0/1986 [00:00<?, ?it/s]

vaw/color.json 77.07


  0%|          | 0/370 [00:00<?, ?it/s]

...

  0%|          | 0/85 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

vg/size.json 62.86


  0%|          | 0/30 [00:10<?, ?it/s]

vg/state.json 72.26
VL-Checklist Attribute: 68.86


## 12 VALSE score


In [17]:
valse = {
    name: load_json(DATA / "VALSE" / "probing_data" / name)
    for name in valse_names
}
print({name: len(items) for name, items in valse.items()})

{'actant-swap.json': 949, 'action-replacement.json': 648, 'relations.json': 535}


In [18]:
valse_names = ['actant-swap.json', 'action-replacement.json', 'relations.json']
valse = {
    name: load_json(DATA / 'VALSE' / 'probing_data' / name)
    for name in valse_names
}

sugar = next((p.parent for p in DATA.rglob('replace_rel.json')), None)
if sugar is None:
    extract('sugar-crepe.tar.gz')
    sugar = folder('replace_rel.json')

vl_images = DATA / 'VL_checklist' / 'VL_checklist_datasets'

valse_scores = {}
for name in valse_names:
    root = sugar if name == 'relations.json' else vl_images
    rows = [
        row(root, x['image_path'], [x['neg_caption'], x['pos_caption']])
        for x in valse[name]
    ]
    valse_scores[name] = evaluate(rows).mean()
    print(name, round(valse_scores[name] * 100, 2))

action = np.mean([valse_scores[name] for name in valse_names[:2]])
record('VALSE Relation', (action + valse_scores['relations.json']) / 2)

Extracted: sugar-crepe.tar.gz


  0%|          | 0/30 [00:00<?, ?it/s]

actant-swap.json 76.71


  0%|          | 0/21 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

action-replacement.json 80.25


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>

  0%|          | 0/17 [00:00<?, ?it/s]


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    Exception ignored in: assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>
AssertionError
: can only test a child processTraceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fe97e7ef100>self._shutdown_workers()

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  

relations.json 71.59
VALSE Relation: 75.03


## 13 Your Table 1 comparison
Blank entries mean a test has not completed. All measurements come from evaluation, not copied targets.


In [20]:
from google.colab import files

today = pd.read_csv('/content/MACCO_Table1.csv', index_col=0)
uploaded = files.upload()
yesterday = pd.read_csv(next(iter(uploaded)), index_col=0)

combined = today.copy()
combined['Our MACCO'] = today['Our MACCO'].combine_first(yesterday['Our MACCO'])
combined['Difference'] = combined['Our MACCO'] - combined['Paper MACCO']

results.update(combined['Our Fine-tuned MACCO'].dropna().to_dict())
combined.to_csv('/content/MACCO_Table1.csv')

display(combined.round(2))
files.download('/content/MACCO_Table1.csv')

StopIteration: 

In [19]:
table1 = show_table()
display(table1.round(2))
table1.to_csv('/content/MACCO_Table1.csv')


,Paper MACCO,Our MACCO,Difference
ARO Relation,73.1,NaN,NaN
ARO Attribute,68.5,NaN,NaN
ARO Order,76.0,NaN,NaN
Sugar-Crepe Relation,77.1,NaN,NaN
Sugar-Crepe Attribute,79.1,NaN,NaN
VL-Checklist Relation,70.2,NaN,NaN
VL-Checklist Attribute,68.7,68.86,0.16
VALSE Relation,75.3,75.03,-0.27
What's-up,43.2,NaN,NaN


[Paper](https://aclanthology.org/2026.acl-long.1490.pdf) · [Model](https://huggingface.co/hiker-lw/MACCO) · [Data](https://huggingface.co/datasets/hiker-lw/VL-Compositionality-Benchmarks)

Scoring follows the paper's category/subset averages, crops and caption ordering. The second VG-QA subset uses VG images, correcting a copy-paste error in the released evaluator. ARO-Order's shuffled captions depend on the seed and language resources. This standalone evaluator uses OpenCLIP 2.32.0 with the checkpoint's GELU model definition; it has not been benchmarked end-to-end against the authors' bundled implementation. Differences must remain reported rather than forced to match.
